# Notebook 01: Data Cleaning & Feature Engineering
Opx ML Thermobarometer
Author: [Your name]
Date: 2026-04-04

This notebook loads the ExPetDB orthopyroxene dataset, cleans and filters it, recalculates cation proportions, and writes cleaned core/full CSV outputs with a provenance log.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (ROOT, DATA_RAW, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
                    MODELS, FIGURES, RESULTS, LOGS, EXPETDB)

import pandas as pd
import numpy as np

In [2]:
INFILE = EXPETDB
OUTDIR = DATA_PROC

OXIDE_TOTAL_MIN = 95.0
OXIDE_TOTAL_MAX = 102.0
CATION_SUM_MIN = 3.95
CATION_SUM_MAX = 4.05
WO_MAX_MOL_PCT = 5.0  # pigeonite filter

OXIDE_INFO = {
    'SiO2':      (60.084,  1, 2),
    'TiO2':      (79.866,  1, 2),
    'Al2O3':     (101.961, 2, 3),
    'Cr2O3':     (151.990, 2, 3),
    'FeO_total': (71.844,  1, 1),
    'MnO':       (70.937,  1, 1),
    'MgO':       (40.304,  1, 1),
    'CaO':       (56.077,  1, 1),
    'Na2O':      (61.979,  2, 1),
}

CATION_NAMES = {
    'SiO2': 'Si', 'TiO2': 'Ti', 'Al2O3': 'Al', 'Cr2O3': 'Cr',
    'FeO_total': 'Fe', 'MnO': 'Mn', 'MgO': 'Mg', 'CaO': 'Ca', 'Na2O': 'Na',
}

RAW_OXIDE_CORE = ['SiO2', 'Al2O3', 'FeO_total', 'MgO', 'CaO']
RAW_OXIDE_FULL = ['SiO2', 'TiO2', 'Al2O3', 'Cr2O3', 'FeO_total', 'MnO', 'MgO', 'CaO', 'Na2O']

log_lines = []

def log(msg: str) -> None:
    print(msg)
    log_lines.append(msg)

## Step 1: Extract and merge

In [3]:
log('STEP 1: Extract and merge')
exp = pd.read_excel(INFILE, sheet_name='Experiment')
opx = pd.read_excel(INFILE, sheet_name='Orthopyroxene')

exp_dedup = exp.drop_duplicates(subset='Experiment', keep='first')
merge_cols = ['Experiment', 'T (C)', 'P (GPa)', 'Device', 'Container', 'Rock type', 'Type']
df = opx.merge(exp_dedup[merge_cols], on='Experiment', how='left')

df['P_kbar'] = df['P (GPa)'] * 10.0
df['T_C'] = df['T (C)']

n0 = len(df)
df = df.dropna(subset=['T_C', 'P_kbar'])
log(f'  Raw opx rows: {n0}')
log(f'  After dropping missing P/T: {len(df)} (dropped {n0 - len(df)})')

STEP 1: Extract and merge


  Raw opx rows: 1796
  After dropping missing P/T: 1796 (dropped 0)


C:\Users\NQTa\AppData\Local\Temp\ipykernel_37504\2128923013.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['P_kbar'] = df['P (GPa)'] * 10.0
C:\Users\NQTa\AppData\Local\Temp\ipykernel_37504\2128923013.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['T_C'] = df['T (C)']


## Step 2: Extract oxides and handle iron

In [4]:
log('\nSTEP 2: Extract oxides, handle iron')
oxide_value_cols = {
    'SiO2': 'SiO2 value', 'TiO2': 'TiO2 value', 'Al2O3': 'Al2O3 value',
    'Cr2O3': 'Cr2O3 value', 'FeO': 'FeO value', 'Fe2O3': 'Fe2O3 value',
    'MnO': 'MnO value', 'MgO': 'MgO value', 'CaO': 'CaO value', 'Na2O': 'Na2O value',
}
for short, full in oxide_value_cols.items():
    df[short] = pd.to_numeric(df[full], errors='coerce')

fe2o3_n = df['Fe2O3'].notna().sum()
df['FeO_total'] = df['FeO'].fillna(0) + df['Fe2O3'].fillna(0) * 0.8998
both_fe_missing = df['FeO'].isna() & df['Fe2O3'].isna()
df.loc[both_fe_missing, 'FeO_total'] = np.nan
log(f'  Fe2O3 reported in {fe2o3_n} rows; all Fe treated as FeO_total')


STEP 2: Extract oxides, handle iron
  Fe2O3 reported in 4 rows; all Fe treated as FeO_total


C:\Users\NQTa\AppData\Local\Temp\ipykernel_37504\2254342555.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[short] = pd.to_numeric(df[full], errors='coerce')
C:\Users\NQTa\AppData\Local\Temp\ipykernel_37504\2254342555.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[short] = pd.to_numeric(df[full], errors='coerce')
C:\Users\NQTa\AppData\Local\Temp\ipykernel_37504\2254342555.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

## Step 3: Oxide total filter

In [5]:
log(f'\nSTEP 3: Oxide total filter ({OXIDE_TOTAL_MIN}-{OXIDE_TOTAL_MAX} wt%)')
df['oxide_total'] = df[RAW_OXIDE_FULL].sum(axis=1, min_count=3)
n_before = len(df)
mask_total = (df['oxide_total'] >= OXIDE_TOTAL_MIN) & (df['oxide_total'] <= OXIDE_TOTAL_MAX)
dropped_total = df[~mask_total & df['oxide_total'].notna()]
df = df[mask_total].copy()
log(f'  Dropped {n_before - len(df)} rows (below {OXIDE_TOTAL_MIN}: {(dropped_total['oxide_total'] < OXIDE_TOTAL_MIN).sum()}, '
    f'above {OXIDE_TOTAL_MAX}: {(dropped_total['oxide_total'] > OXIDE_TOTAL_MAX).sum()}, '
    f'null: {(~mask_total & df['oxide_total'].isna() if 'oxide_total' in df.columns else pd.Series(dtype=bool)).sum()})')
log(f'  Remaining: {len(df)}')


STEP 3: Oxide total filter (95.0-102.0 wt%)
  Dropped 67 rows (below 95.0: 6, above 102.0: 18, null: 0)
  Remaining: 1729


C:\Users\NQTa\AppData\Local\Temp\ipykernel_37504\2867960258.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['oxide_total'] = df[RAW_OXIDE_FULL].sum(axis=1, min_count=3)


## Step 4: Cation recalculation

In [6]:
log('\nSTEP 4: Cation recalculation (6-oxygen basis)')
for ox, (mw, ncat, nox) in OXIDE_INFO.items():
    df[f'{ox}_moles'] = df[ox].fillna(0) / mw
    df[f'{ox}_oxy'] = df[f'{ox}_moles'] * nox

df['total_oxy'] = sum(df[f'{ox}_oxy'] for ox in OXIDE_INFO)
df['norm_factor'] = 6.0 / df['total_oxy']

for ox, (mw, ncat, nox) in OXIDE_INFO.items():
    cat = CATION_NAMES[ox]
    df[f'cat_{cat}'] = df[f'{ox}_moles'] * ncat * df['norm_factor']

cat_cols = [f'cat_{c}' for c in CATION_NAMES.values()]
df['cation_sum'] = df[cat_cols].sum(axis=1)
log(f'  Cation sum: mean={df['cation_sum'].mean():.4f}, '
    f'std={df['cation_sum'].std():.4f}, '
    f'range=[{df['cation_sum'].min():.4f}, {df['cation_sum'].max():.4f}]')


STEP 4: Cation recalculation (6-oxygen basis)


  Cation sum: mean=4.0007, std=0.0181, range=[3.7408, 4.2655]


## Step 5: Core oxide and cation sum filters

In [7]:
log(f'\nSTEP 5: Core oxide requirement + cation sum filter ({CATION_SUM_MIN}-{CATION_SUM_MAX})')
n_before = len(df)
core_present = df[['SiO2', 'MgO', 'FeO_total']].notna().all(axis=1)
df = df[core_present].copy()
log(f'  After requiring SiO2+MgO+FeO: {len(df)} (dropped {n_before - len(df)})')

n_before = len(df)
df = df[(df['cation_sum'] >= CATION_SUM_MIN) & (df['cation_sum'] <= CATION_SUM_MAX)].copy()
log(f'  After cation sum filter: {len(df)} (dropped {n_before - len(df)})')


STEP 5: Core oxide requirement + cation sum filter (3.95-4.05)
  After requiring SiO2+MgO+FeO: 1497 (dropped 232)
  After cation sum filter: 1481 (dropped 16)


## Step 6: Pigeonite filter

In [8]:
log(f'\nSTEP 6: Pigeonite filter (Wo <= {WO_MAX_MOL_PCT} mol%)')
df['sum_MgFeCa'] = df['cat_Mg'] + df['cat_Fe'] + df['cat_Ca']
df['En'] = df['cat_Mg'] / df['sum_MgFeCa'] * 100
df['Fs'] = df['cat_Fe'] / df['sum_MgFeCa'] * 100
df['Wo'] = df['cat_Ca'] / df['sum_MgFeCa'] * 100

n_pigeonite = (df['Wo'] > WO_MAX_MOL_PCT).sum()
n_before = len(df)
df = df[df['Wo'] <= WO_MAX_MOL_PCT].copy()
log(f'  Dropped {n_before - len(df)} analyses with Wo > {WO_MAX_MOL_PCT}%')
log(f'  Remaining: {len(df)}')


STEP 6: Pigeonite filter (Wo <= 5.0 mol%)


  Dropped 133 analyses with Wo > 5.0%
  Remaining: 1348


In [9]:
log(f'\nSTEP 6b: Pressure ceiling filter (P <= 100 kbar)')
n_before = len(df)
df = df[df['P_kbar'] <= 100.0].copy()
log(f'  Dropped {n_before - len(df)} rows with P > 100 kbar (opx unstable)')
log(f'  Remaining: {len(df)}')
log(f'  P range after filter: {df["P_kbar"].min():.1f}-{df["P_kbar"].max():.1f} kbar')


STEP 6b: Pressure ceiling filter (P <= 100 kbar)
  Dropped 29 rows with P > 100 kbar (opx unstable)
  Remaining: 1319
  P range after filter: 0.0-100.0 kbar


## Step 7: Feature engineering

In [10]:
log('\nSTEP 7: Feature engineering')
df['Mg_num'] = df['cat_Mg'] / (df['cat_Mg'] + df['cat_Fe'])
df['Al_IV'] = np.clip(2.0 - df['cat_Si'], 0, df['cat_Al'])
df['Al_VI'] = df['cat_Al'] - df['Al_IV']
df['MgTs'] = df['Al_IV']
df['En_frac'] = df['cat_Mg'] / df['sum_MgFeCa']
df['Fs_frac'] = df['cat_Fe'] / df['sum_MgFeCa']
df['Wo_frac'] = df['cat_Ca'] / df['sum_MgFeCa']

log(f'  Mg#:   [{df['Mg_num'].min():.3f}, {df['Mg_num'].max():.3f}], median={df['Mg_num'].median():.3f}')
log(f'  Al_IV: [{df['Al_IV'].min():.4f}, {df['Al_IV'].max():.4f}], median={df['Al_IV'].median():.4f}')
log(f'  Al_VI: [{df['Al_VI'].min():.4f}, {df['Al_VI'].max():.4f}], median={df['Al_VI'].median():.4f}')


STEP 7: Feature engineering
  Mg#:   [0.086, 1.000], median=0.884
  Al_IV: [0.0000, 0.3602], median=0.0741
  Al_VI: [0.0000, 0.3333], median=0.0549


## Step 8: Save datasets

In [11]:
log('\nSTEP 8: Save datasets')
prov_cols = ['Experiment', 'Citation', 'DOI']
if 'DOI' not in df.columns:
    df['DOI'] = np.nan
target_cols = ['T_C', 'P_kbar']
cation_feature_cols = ['cat_Si', 'cat_Ti', 'cat_Al', 'cat_Cr', 'cat_Fe', 'cat_Mn', 'cat_Mg', 'cat_Ca', 'cat_Na']
eng_cols = ['Mg_num', 'Al_IV', 'Al_VI', 'En_frac', 'Fs_frac', 'Wo_frac', 'MgTs']
meta_cols = ['oxide_total', 'cation_sum', 'Device', 'Rock type']

output_cols = prov_cols + target_cols + RAW_OXIDE_FULL + cation_feature_cols + eng_cols + meta_cols
output_cols = [c for c in output_cols if c in df.columns]

df_core = df[df[RAW_OXIDE_CORE].notna().all(axis=1)].copy()
df_full = df[df[RAW_OXIDE_FULL].notna().all(axis=1)].copy()

df_core[output_cols].to_csv(OUTDIR / 'opx_clean_core.csv', index=False)
df_full[output_cols].to_csv(OUTDIR / 'opx_clean_full.csv', index=False)

log(f'  Core dataset: {len(df_core)} rows, {df_core["Experiment"].nunique()} experiments, {df_core["Citation"].nunique()} citations')
log(f'  Full dataset: {len(df_full)} rows, {df_full["Experiment"].nunique()} experiments, {df_full["Citation"].nunique()} citations')
log(f'  Core P range: {df_core["P_kbar"].min():.1f}-{df_core["P_kbar"].max():.1f} kbar')
log(f'  Core T range: {df_core["T_C"].min():.0f}-{df_core["T_C"].max():.0f} C')

assert df_core['P_kbar'].max() <= 100.0, f'P_kbar max {df_core["P_kbar"].max()} exceeds 100'
assert df_full['P_kbar'].max() <= 100.0, f'P_kbar max {df_full["P_kbar"].max()} exceeds 100'
log(f'  Verified: P_kbar max <= 100 kbar in both datasets')

with open(LOGS / 'cleaning_log.txt', 'w') as f:
    f.write('\n'.join(log_lines))

log(f'\nDone. CSVs saved to {OUTDIR}, log saved to {LOGS}')


STEP 8: Save datasets

  Core dataset: 1148 rows, 1034 experiments, 126 citations
  Full dataset: 525 rows, 493 experiments, 68 citations
  Core P range: 0.0-100.0 kbar
  Core T range: 550-1850 C
  Verified: P_kbar max <= 100 kbar in both datasets

Done. CSVs saved to C:\Users\NQTa\Documents\MLCourse\Final Project\data\processed, log saved to C:\Users\NQTa\Documents\MLCourse\Final Project\logs
